In [5]:
import pandas as pd
import numpy as np

print("Final Prediction Notebook Ready")

Final Prediction Notebook Ready


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/Telco-Customer-Churn-cleaned.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
import joblib

# Load trained model
xgb_model = joblib.load("../models/xgb_model.pkl")

# Load preprocessing pipeline
preprocessor = joblib.load("../models/preprocessor.pkl")

print("Model loaded successfully!")
print("Preprocessor loaded successfully!")

Model loaded successfully!
Preprocessor loaded successfully!


In [8]:
CHURN_THRESHOLD = 0.35

print("Final churn threshold:", CHURN_THRESHOLD)

Final churn threshold: 0.35


In [9]:
# Create a copy for prediction
prediction_data = df.copy()

# Convert TotalCharges to numeric
prediction_data["TotalCharges"] = pd.to_numeric(
    prediction_data["TotalCharges"],
    errors="coerce"
)

# Fill any missing TotalCharges
prediction_data["TotalCharges"] = prediction_data["TotalCharges"].fillna(0)

print("Prediction data shape:", prediction_data.shape)

Prediction data shape: (7043, 21)


In [10]:
prediction_data["AverageMonthlySpend"] = (
    prediction_data["TotalCharges"] /
    prediction_data["tenure"].replace(0, np.nan)
).fillna(
    prediction_data["MonthlyCharges"]
)

service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

prediction_data["ServiceCount"] = (
    prediction_data[service_columns]
    .apply(lambda row: (row == "Yes").sum(), axis=1)
)

prediction_data["HasTechSupport"] = (
    prediction_data["TechSupport"] == "Yes"
).astype(int)

print(
    prediction_data[
        [
            "customerID",
            "AverageMonthlySpend",
            "ServiceCount",
            "HasTechSupport"
        ]
    ].head()
)

   customerID  AverageMonthlySpend  ServiceCount  HasTechSupport
0  7590-VHVEG            29.850000             1               0
1  5575-GNVDE            55.573529             3               0
2  3668-QPYBK            54.075000             3               0
3  7795-CFOCW            40.905556             3               1
4  9237-HQITU            75.825000             1               0


In [11]:
X_prediction = prediction_data.drop(
    columns=["customerID", "Churn"]
)

In [13]:
X_prediction_processed = preprocessor.transform(
    X_prediction
)

print(
    "Processed prediction data:",
    X_prediction_processed.shape
)

ValueError: columns are missing: {'TenureGroup'}

In [14]:
prediction_data["TenureGroup"] = pd.cut(
    prediction_data["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

print(prediction_data["TenureGroup"].value_counts())

TenureGroup
49-72 months    2239
0-12 months     2186
25-48 months    1594
13-24 months    1024
Name: count, dtype: int64


In [15]:
X_prediction = prediction_data.drop(
    columns=["customerID", "Churn"]
)

In [16]:
X_prediction_processed = preprocessor.transform(
    X_prediction
)

print(
    "Processed prediction data:",
    X_prediction_processed.shape
)

Processed prediction data: (7043, 36)


In [17]:
# Generate churn probabilities
churn_probabilities = xgb_model.predict_proba(
    X_prediction_processed
)[:, 1]

print("First 10 churn probabilities:")
print(churn_probabilities[:10])

First 10 churn probabilities:
[0.7202053  0.03038575 0.3698431  0.03959299 0.66115427 0.91866726
 0.41764697 0.15247989 0.53699714 0.02317204]


In [18]:
# Convert probabilities into churn predictions
churn_predictions = (
    churn_probabilities >= CHURN_THRESHOLD
).astype(int)

print("Churn predictions:")
print(churn_predictions[:10])

Churn predictions:
[1 0 1 0 1 1 1 0 1 0]


In [19]:
print(
    "Predicted churn customers:",
    churn_predictions.sum()
)

print(
    "Total customers:",
    len(churn_predictions)
)

Predicted churn customers: 2306
Total customers: 7043


In [20]:
risk_table = pd.DataFrame({
    "CustomerID": prediction_data["customerID"],
    "ChurnProbability": churn_probabilities,
    "PredictedChurn": churn_predictions
})

risk_table["RiskLevel"] = risk_table["ChurnProbability"].apply(
    assign_risk
)

risk_table.head(10)

NameError: name 'assign_risk' is not defined

In [21]:
def assign_risk(probability):
    if probability >= 0.70:
        return "High Risk"
    elif probability >= 0.35:
        return "Medium Risk"
    else:
        return "Low Risk"

In [22]:
risk_table = pd.DataFrame({
    "CustomerID": prediction_data["customerID"],
    "ChurnProbability": churn_probabilities,
    "PredictedChurn": churn_predictions
})

risk_table["RiskLevel"] = risk_table["ChurnProbability"].apply(
    assign_risk
)

risk_table.head(10)

,CustomerID,ChurnProbability,PredictedChurn,RiskLevel
0,7590-VHVEG,0.720205,1,High Risk
1,5575-GNVDE,0.030386,0,Low Risk
2,3668-QPYBK,0.369843,1,Medium Risk
3,7795-CFOCW,0.039593,0,Low Risk
4,9237-HQITU,0.661154,1,Medium Risk
5,9305-CDSKC,0.918667,1,High Risk
6,1452-KIOVK,0.417647,1,Medium Risk
7,6713-OKOMC,0.152480,0,Low Risk
8,7892-POOKP,0.536997,1,Medium Risk
9,6388-TABGU,0.023172,0,Low Risk


In [23]:
print(risk_table["RiskLevel"].value_counts())

RiskLevel
Low Risk       4737
Medium Risk    1674
High Risk       632
Name: count, dtype: int64


In [24]:
print(
    risk_table["RiskLevel"].value_counts(normalize=True) * 100
)

RiskLevel
Low Risk       67.258271
Medium Risk    23.768281
High Risk       8.973449
Name: proportion, dtype: float64


In [25]:
risk_table["MonthlyCharges"] = prediction_data["MonthlyCharges"].values

In [26]:
high_risk_customers = risk_table[
    risk_table["RiskLevel"] == "High Risk"
]

predicted_churn_customers = risk_table[
    risk_table["PredictedChurn"] == 1
]

print("High-risk customers:", len(high_risk_customers))
print("Predicted churn customers:", len(predicted_churn_customers))

High-risk customers: 632
Predicted churn customers: 2306


In [27]:
revenue_at_risk = predicted_churn_customers["MonthlyCharges"].sum()

print(
    "Estimated monthly revenue at risk: $",
    round(revenue_at_risk, 2)
)

Estimated monthly revenue at risk: $ 174705.7


In [28]:
high_risk_revenue = high_risk_customers["MonthlyCharges"].sum()

print(
    "High-risk monthly revenue at risk: $",
    round(high_risk_revenue, 2)
)

High-risk monthly revenue at risk: $ 51870.5


In [29]:
top_risk_customers = risk_table.sort_values(
    "ChurnProbability",
    ascending=False
).head(20)

top_risk_customers


,CustomerID,ChurnProbability,PredictedChurn,RiskLevel,MonthlyCharges
2208,7216-EWTRS,0.941327,1,High Risk,100.80
4517,2012-NWRPA,0.931918,1,High Risk,99.55
4800,9300-AGZNL,0.929708,1,High Risk,94.00
1976,9497-QCMMS,0.929299,1,High Risk,93.55
3380,5178-LMXOP,0.925648,1,High Risk,95.10
4459,3178-FESZO,0.922401,1,High Risk,100.25
4585,1069-XAIEM,0.922250,1,High Risk,85.05
2577,4910-GMJOT,0.919094,1,High Risk,94.60
5,9305-CDSKC,0.918667,1,High Risk,99.65
6089,8775-LHDJH,0.917515,1,High Risk,90.60


In [30]:
top_risk_customers = prediction_data[
    [
        "customerID",
        "tenure",
        "MonthlyCharges",
        "Contract",
        "InternetService",
        "TechSupport",
        "PaymentMethod"
    ]
].merge(
    risk_table[
        [
            "CustomerID",
            "ChurnProbability",
            "RiskLevel"
        ]
    ],
    left_on="customerID",
    right_on="CustomerID"
).sort_values(
    "ChurnProbability",
    ascending=False
).head(20)

top_risk_customers

,customerID,tenure,MonthlyCharges,Contract,InternetService,TechSupport,PaymentMethod,CustomerID,ChurnProbability,RiskLevel
2208,7216-EWTRS,1,100.80,Month-to-month,Fiber optic,No,Electronic check,7216-EWTRS,0.941327,High Risk
4517,2012-NWRPA,11,99.55,Month-to-month,Fiber optic,No,Electronic check,2012-NWRPA,0.931918,High Risk
4800,9300-AGZNL,1,94.00,Month-to-month,Fiber optic,No,Electronic check,9300-AGZNL,0.929708,High Risk
1976,9497-QCMMS,1,93.55,Month-to-month,Fiber optic,No,Electronic check,9497-QCMMS,0.929299,High Risk
3380,5178-LMXOP,1,95.10,Month-to-month,Fiber optic,No,Electronic check,5178-LMXOP,0.925648,High Risk
4459,3178-FESZO,1,100.25,Month-to-month,Fiber optic,No,Credit card (automatic),3178-FESZO,0.922401,High Risk
4585,1069-XAIEM,1,85.05,Month-to-month,Fiber optic,No,Electronic check,1069-XAIEM,0.922250,High Risk
2577,4910-GMJOT,1,94.60,Month-to-month,Fiber optic,No,Electronic check,4910-GMJOT,0.919094,High Risk
5,9305-CDSKC,8,99.65,Month-to-month,Fiber optic,No,Electronic check,9305-CDSKC,0.918667,High Risk
6089,8775-LHDJH,1,90.60,Month-to-month,Fiber optic,No,Electronic check,8775-LHDJH,0.917515,High Risk


In [31]:
top_risk_customers["ChurnProbability"] = (
    top_risk_customers["ChurnProbability"] * 100
).round(2)

top_risk_customers

,customerID,tenure,MonthlyCharges,Contract,InternetService,TechSupport,PaymentMethod,CustomerID,ChurnProbability,RiskLevel
2208,7216-EWTRS,1,100.80,Month-to-month,Fiber optic,No,Electronic check,7216-EWTRS,94.129997,High Risk
4517,2012-NWRPA,11,99.55,Month-to-month,Fiber optic,No,Electronic check,2012-NWRPA,93.190002,High Risk
4800,9300-AGZNL,1,94.00,Month-to-month,Fiber optic,No,Electronic check,9300-AGZNL,92.970001,High Risk
1976,9497-QCMMS,1,93.55,Month-to-month,Fiber optic,No,Electronic check,9497-QCMMS,92.930000,High Risk
3380,5178-LMXOP,1,95.10,Month-to-month,Fiber optic,No,Electronic check,5178-LMXOP,92.559998,High Risk
4459,3178-FESZO,1,100.25,Month-to-month,Fiber optic,No,Credit card (automatic),3178-FESZO,92.239998,High Risk
4585,1069-XAIEM,1,85.05,Month-to-month,Fiber optic,No,Electronic check,1069-XAIEM,92.220001,High Risk
2577,4910-GMJOT,1,94.60,Month-to-month,Fiber optic,No,Electronic check,4910-GMJOT,91.910004,High Risk
5,9305-CDSKC,8,99.65,Month-to-month,Fiber optic,No,Electronic check,9305-CDSKC,91.870003,High Risk
6089,8775-LHDJH,1,90.60,Month-to-month,Fiber optic,No,Electronic check,8775-LHDJH,91.750000,High Risk


In [32]:
def recommend_action(row):
    if row["RiskLevel"] == "High Risk":
        if row["Contract"] == "Month-to-month":
            return "Offer annual contract discount + proactive support"
        elif row["MonthlyCharges"] >= 80:
            return "Offer personalized discount + premium support"
        else:
            return "Priority retention outreach + support follow-up"

    elif row["RiskLevel"] == "Medium Risk":
        if row["Contract"] == "Month-to-month":
            return "Offer contract upgrade incentive"
        else:
            return "Monitor customer + targeted engagement"

    else:
        return "Continue regular engagement"

In [33]:
risk_table = risk_table.merge(
    prediction_data[
        [
            "customerID",
            "tenure",
            "MonthlyCharges",
            "Contract",
            "InternetService",
            "TechSupport",
            "PaymentMethod"
        ]
    ],
    left_on="CustomerID",
    right_on="customerID",
    how="left"
)

risk_table["RecommendedAction"] = risk_table.apply(
    recommend_action,
    axis=1
)

In [34]:
risk_table[
    [
        "CustomerID",
        "ChurnProbability",
        "RiskLevel",
        "MonthlyCharges",
        "Contract",
        "RecommendedAction"
    ]
].head(10)

KeyError: "['MonthlyCharges'] not in index"

In [35]:
risk_table["MonthlyCharges"] = prediction_data["MonthlyCharges"].values

In [36]:
print(risk_table.columns.tolist())

['CustomerID', 'ChurnProbability', 'PredictedChurn', 'RiskLevel', 'MonthlyCharges_x', 'customerID', 'tenure', 'MonthlyCharges_y', 'Contract', 'InternetService', 'TechSupport', 'PaymentMethod', 'RecommendedAction', 'MonthlyCharges']


In [37]:
# Keep the correct MonthlyCharges column
risk_table["MonthlyCharges"] = risk_table["MonthlyCharges_x"]

# Remove duplicate/unnecessary columns
risk_table = risk_table.drop(
    columns=["MonthlyCharges_x", "MonthlyCharges_y"],
    errors="ignore"
)

# Remove duplicate customerID column
risk_table = risk_table.drop(
    columns=["customerID"],
    errors="ignore"
)

print(risk_table.columns.tolist())

['CustomerID', 'ChurnProbability', 'PredictedChurn', 'RiskLevel', 'tenure', 'Contract', 'InternetService', 'TechSupport', 'PaymentMethod', 'RecommendedAction', 'MonthlyCharges']


In [38]:
risk_table[
    [
        "CustomerID",
        "ChurnProbability",
        "RiskLevel",
        "MonthlyCharges",
        "Contract",
        "RecommendedAction"
    ]
].head(10)

,CustomerID,ChurnProbability,RiskLevel,MonthlyCharges,Contract,RecommendedAction
0,7590-VHVEG,0.720205,High Risk,29.85,Month-to-month,Offer annual contract discount + proactive sup...
1,5575-GNVDE,0.030386,Low Risk,56.95,One year,Continue regular engagement
2,3668-QPYBK,0.369843,Medium Risk,53.85,Month-to-month,Offer contract upgrade incentive
3,7795-CFOCW,0.039593,Low Risk,42.30,One year,Continue regular engagement
4,9237-HQITU,0.661154,Medium Risk,70.70,Month-to-month,Offer contract upgrade incentive
5,9305-CDSKC,0.918667,High Risk,99.65,Month-to-month,Offer annual contract discount + proactive sup...
6,1452-KIOVK,0.417647,Medium Risk,89.10,Month-to-month,Offer contract upgrade incentive
7,6713-OKOMC,0.152480,Low Risk,29.75,Month-to-month,Continue regular engagement
8,7892-POOKP,0.536997,Medium Risk,104.80,Month-to-month,Offer contract upgrade incentive
9,6388-TABGU,0.023172,Low Risk,56.15,One year,Continue regular engagement
